<a href="https://colab.research.google.com/github/rm571222/dataholics-oracle-challenge/blob/main/notebooks/01_data_exploration/nb3_data_exploration_cnes_json.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB3 — Data Exploration: Cadastro Complementar de Estabelecimentos (CNES, JSON)

**Projeto DATAHOLICS — FIAP Challenge | Parceria Oracle**

Este notebook documenta a exploração da terceira fonte primária do projeto: o cadastro completo de estabelecimentos de saúde do CNES, disponibilizado em **formato JSON**. Esta fonte é a peça planejada para demonstrar, na arquitetura final, a convivência entre dado relacional e documento semiestruturado.

## Fonte de dados

- **Sistema:** CNES (Cadastro Nacional de Estabelecimentos de Saúde) — módulo de Estabelecimentos
- **Publicação:** Ministério da Saúde, Portal de Dados Abertos
- **URL:** https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/CNES/cnes_estabelecimentos_json.zip
- **Formato:** JSON — um documento por estabelecimento, com estrutura potencialmente variável entre tipos de unidade

## Estrutura deste notebook

1. Exploração inicial dos dados brutos
2. Seleção das principais variáveis + exploração/qualidade
3. Colunas de código sem descrição e domínios observados
4. Esclarecimento de inconsistências encontradas
5. Cruzamento de validação com a fonte SIH/pysus (NB1)

## 1. Exploração inicial dos dados brutos

In [ ]:
import requests, zipfile, io, json
import pandas as pd

resp = requests.get("https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/CNES/cnes_estabelecimentos_json.zip")
z = zipfile.ZipFile(io.BytesIO(resp.content))

with z.open(z.namelist()[0]) as f:
    dados_cnes = json.load(f)

df_cnes = pd.DataFrame(dados_cnes)
print(f"Total de estabelecimentos no Brasil: {len(df_cnes)}")

df_cnes_sp = df_cnes[df_cnes['CO_UF'] == '35'].copy()
print(f"Total de estabelecimentos em SP: {len(df_cnes_sp)}")

df_cnes_sp.info()

Total de estabelecimentos no Brasil: 634034
Total de estabelecimentos em SP: 153366
<class 'pandas.core.frame.DataFrame'>
Index: 153366 entries, 234 to 634030
Data columns (total 36 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   CO_CNES                   153366 non-null  object
 1   CO_UNIDADE                153366 non-null  object
 2   CO_UF                     153366 non-null  object
 3   CO_IBGE                   153366 non-null  object
 4   NU_CNPJ_MANTENEDORA       153366 non-null  object
 5   NO_RAZAO_SOCIAL           153366 non-null  object
 6   NO_FANTASIA               153366 non-null  object
 7   CO_NATUREZA_ORGANIZACAO   153366 non-null  object
 8   DS_NATUREZA_ORGANIZACAO   153366 non-null  object
 9   TP_GESTAO                 153366 non-null  object
 10  CO_NIVEL_HIERARQUIA       153366 non-null  object
 11  DS_NIVEL_HIERARQUIA       153366 non-null  object
 12  CO_ESFERA_ADMINISTRATIVA  153366 

In [ ]:
df_cnes_sp.head()

,CO_CNES,CO_UNIDADE,CO_UF,CO_IBGE,NU_CNPJ_MANTENEDORA,NO_RAZAO_SOCIAL,NO_FANTASIA,CO_NATUREZA_ORGANIZACAO,DS_NATUREZA_ORGANIZACAO,TP_GESTAO,...,NO_EMAIL,CO_NATUREZA_JUR,ST_CENTRO_CIRURGICO,ST_CENTRO_OBSTETRICO,ST_CENTRO_NEONATAL,ST_ATEND_HOSPITALAR,ST_SERVICO_APOIO,ST_ATEND_AMBULATORIAL,CO_MOTIVO_DESAB,CO_AMBULATORIAL_SUS
234,3204,3518800003204,35,351880,,ARVP CLINICA ODONTOLOGICA LTDA,OESP PRIME,,,M,...,oespprime.guarulhos@gmail.com,2062,0.0,0.0,0.0,0.0,0.0,0.0,08,NAO
271,3735,3518800003735,35,351880,,MODATO ODONTOLOGIA LTDA,MODATO ODONTOLOGIA,,,M,...,modatoodontologia@gmail.com,2062,0.0,0.0,0.0,0.0,0.0,0.0,,NAO
285,3905,3518800003905,35,351880,,SANDRA REGINA FERREIRA,FERREIRA MARCONI ODONTOLOGIA,,,M,...,ferreiramarconiodonto@gmail.com,2135,0.0,0.0,0.0,0.0,0.0,0.0,,NAO
307,4189,3518800004189,35,351880,,KMG ODONTO CONSULTORIO ODONTOLOGICO EIRELI,KMG ODONTO,,,M,...,kmgodonto@gmail.com,2062,0.0,0.0,0.0,0.0,0.0,0.0,06,NAO
309,4200,3518800004200,35,351880,,FBVALLE ODONTOLOGIA INTEGRADA LTDA,DOUTOR QUALITY ODONTOLOGIA INTEGRADA,,,M,...,evandro@doutorquality.com.br,2062,0.0,0.0,0.0,0.0,0.0,0.0,,NAO


## 2. Seleção das principais variáveis + exploração/qualidade

Das 36 colunas disponíveis, focamos nas que trazem identificação, localização e atributos operacionais do estabelecimento — sem necessidade de definir uma estrutura relacional fixa, já que o valor de manter isso como documento JSON é justamente preservar o registro como vem da fonte.

In [ ]:
colunas_relevantes = ['CO_CNES', 'CO_IBGE', 'NO_FANTASIA', 'NO_RAZAO_SOCIAL',
                       'DS_ESFERA_ADMINISTRATIVA', 'TP_GESTAO', 'TP_UNIDADE',
                       'DS_TURNO_ATENDIMENTO', 'ST_ATEND_HOSPITALAR',
                       'CO_AMBULATORIAL_SUS', 'NU_LATITUDE', 'NU_LONGITUDE']

resumo = pd.DataFrame({
    'pct_nulos': (df_cnes_sp[colunas_relevantes].isna().mean() * 100).round(2),
    'valores_distintos': df_cnes_sp[colunas_relevantes].nunique()
})
print(resumo)
print(f"\nDuplicatas de CO_CNES: {df_cnes_sp['CO_CNES'].duplicated().sum()}")

                          pct_nulos  valores_distintos
CO_CNES                         0.0             153366
CO_IBGE                         0.0                645
NO_FANTASIA                     0.0             145171
NO_RAZAO_SOCIAL                 0.0             129016
DS_ESFERA_ADMINISTRATIVA        0.0                  4
TP_GESTAO                       0.0                  4
TP_UNIDADE                      0.0                 39
DS_TURNO_ATENDIMENTO            0.0                  8
ST_ATEND_HOSPITALAR             0.0                  3
CO_AMBULATORIAL_SUS             0.0                  2
NU_LATITUDE                     0.0             103395
NU_LONGITUDE                    0.0             104496

Duplicatas de CO_CNES: 0


## 3. Colunas de código sem descrição e domínios observados

In [ ]:
print("DS_ESFERA_ADMINISTRATIVA:")
print(df_cnes_sp['DS_ESFERA_ADMINISTRATIVA'].value_counts())

print("\nTP_GESTAO:")
print(df_cnes_sp['TP_GESTAO'].value_counts())

print("\nST_ATEND_HOSPITALAR (indica se é hospital de fato):")
print(df_cnes_sp['ST_ATEND_HOSPITALAR'].apply(lambda x: repr(x)).value_counts())

print("\nCO_AMBULATORIAL_SUS:")
print(df_cnes_sp['CO_AMBULATORIAL_SUS'].value_counts())

DS_ESFERA_ADMINISTRATIVA:
DS_ESFERA_ADMINISTRATIVA
MUNICIPAL    152425
ESTADUAL        807
DUPLA           100
SEM              34
Name: count, dtype: int64

TP_GESTAO:
TP_GESTAO
M    152425
E       807
D       100
S        34
Name: count, dtype: int64

ST_ATEND_HOSPITALAR (indica se é hospital de fato):
ST_ATEND_HOSPITALAR
'0.0'    142962
''         9009
'1.0'      1395
Name: count, dtype: int64

CO_AMBULATORIAL_SUS:
CO_AMBULATORIAL_SUS
NAO    139887
SIM     13479
Name: count, dtype: int64


Diferente do que se observou no cadastro de Leitos (NB2), aqui `TP_GESTAO` tem **4 valores** — `M`, `E`, `D` e `S` — sendo o `D` (Dupla) e o `S` valores que não haviam aparecido antes. Isso motivou a investigação da Seção 4.

## 4. Esclarecimento de inconsistências encontradas

In [ ]:
# Investigação do código TP_GESTAO = 'S'
tp_gestao_s = df_cnes_sp[df_cnes_sp['TP_GESTAO'] == 'S']
print(f"Total TP_GESTAO='S': {len(tp_gestao_s)}")
print(f"Com CO_MOTIVO_DESAB preenchido: {(tp_gestao_s['CO_MOTIVO_DESAB'].astype(str).str.strip() != '').sum()}")
print("\n=> Confirmado: TP_GESTAO='S' corresponde a estabelecimentos DESABILITADOS no cadastro (100% dos casos).")

Total TP_GESTAO='S': 34
Com CO_MOTIVO_DESAB preenchido: 34

=> Confirmado: TP_GESTAO='S' corresponde a estabelecimentos DESABILITADOS no cadastro (100% dos casos).


In [ ]:
# Investigação do ST_ATEND_HOSPITALAR vazio
atend_vazio = df_cnes_sp[df_cnes_sp['ST_ATEND_HOSPITALAR'] == '']
print(f"Total ST_ATEND_HOSPITALAR vazio: {len(atend_vazio)} ({len(atend_vazio)/len(df_cnes_sp)*100:.2f}% da base)")

desabilitados = atend_vazio[atend_vazio['CO_MOTIVO_DESAB'].astype(str).str.strip() != '']
nao_desabilitados = atend_vazio[atend_vazio['CO_MOTIVO_DESAB'].astype(str).str.strip() == '']
print(f"  Com motivo de desabilitação: {len(desabilitados)}")
print(f"  Sem motivo de desabilitação: {len(nao_desabilitados)}")

print("\nTipos de unidade mais comuns entre os 'vazios sem desabilitação':")
print(nao_desabilitados.groupby('TP_UNIDADE')['NO_FANTASIA'].apply(lambda x: x.head(2).tolist()).head(10))

Total ST_ATEND_HOSPITALAR vazio: 9009 (5.87% da base)
  Com motivo de desabilitação: 2203
  Sem motivo de desabilitação: 6806

Tipos de unidade mais comuns entre os 'vazios sem desabilitação':
TP_UNIDADE
1                     [DIVISAO DE TRANSPORTE SANITARIO]
16                                     [IMD SANTANA PA]
2     [SERVICO DE ATENCAO DOMICILIAR CAMPO LIMPO, VI...
20                       [PRONTO ATENDIMENTO MUNICIPAL]
22    [CONS ODONTO CRISTIANE BARBOSA DA SILVEIRA MON...
36    [ORTOCITY SAO MIGUEL, CLINICA DE RECUPERACAO R...
39            [GUACELLI, BERRINI CENTRO DE DIAGNOSTICO]
4                 [APAE PALMARES PAULISTA, MAMA IMAGEM]
40    [CRUZ BRANCA, TRANSPORTE SANITARIO ELETIVO CEN...
42             [SAMU 1 BASE SEDE 2, SAMU 1 BASE SEDE 6]
Name: NO_FANTASIA, dtype: object


### Conclusão da investigação

Os 9.009 registros com `ST_ATEND_HOSPITALAR` vazio se explicam por duas causas legítimas, nenhuma delas erro de dado:

| Subgrupo | Quantidade | Explicação |
|---|---|---|
| Estabelecimentos desabilitados (`CO_MOTIVO_DESAB` preenchido) | 2.203 (24,4%) | Fora de operação no cadastro CNES |
| Tipo de unidade não hospitalar por natureza | 6.806 (75,6%) | Bases do SAMU, farmácias, laboratórios, vigilância sanitária — o campo simplesmente não se aplica a esses tipos de estabelecimento |

Nenhum desses casos interfere na análise principal do projeto, já que o cruzamento com a Tabela de Hospitais (NB2) e com o SIH (NB1) naturalmente filtra apenas estabelecimentos que efetivamente geraram internação.

In [ ]:
# Consistência de geolocalização (faixa aproximada de SP)
df_cnes_sp['NU_LATITUDE_num'] = pd.to_numeric(df_cnes_sp['NU_LATITUDE'], errors='coerce')
df_cnes_sp['NU_LONGITUDE_num'] = pd.to_numeric(df_cnes_sp['NU_LONGITUDE'], errors='coerce')

fora_faixa = df_cnes_sp[
    (df_cnes_sp['NU_LATITUDE_num'] < -25) | (df_cnes_sp['NU_LATITUDE_num'] > -19) |
    (df_cnes_sp['NU_LONGITUDE_num'] < -53) | (df_cnes_sp['NU_LONGITUDE_num'] > -44)
]
print(f"Registros com lat/long fora da faixa esperada de SP: {len(fora_faixa)} (residual, não tratado nesta fase)")
print(f"Registros com lat/long zerado ou nulo: {((df_cnes_sp['NU_LATITUDE_num'] == 0) | df_cnes_sp['NU_LATITUDE_num'].isna()).sum()} (12% da base — limitação conhecida, sem impacto na análise atual, que não usa mapa)")

Registros com lat/long fora da faixa esperada de SP: 29 (residual, não tratado nesta fase)
Registros com lat/long zerado ou nulo: 18525 (12% da base — limitação conhecida, sem impacto na análise atual, que não usa mapa)


## 5. Cruzamento de validação com a fonte SIH/pysus (NB1)

In [ ]:
!pip install -q pysus

from pysus import sih
import pandas as pd

MESES_AMOSTRA = [(2026, m) for m in range(1, 7)]  # jan a jun/2026

dfs_amostra = []
for ano, mes in MESES_AMOSTRA:
    try:
        caminhos = sih(state="SP", year=ano, month=mes)
        caminhos_rd = [p for p in caminhos if "RDSP" in p.upper()]
        if caminhos_rd:
            dfs_amostra.append(pd.concat([pd.read_parquet(p) for p in caminhos_rd], ignore_index=True))
    except Exception as e:
        print(f'{ano}-{mes:02d} falhou: {e}')

df_sih = pd.concat(dfs_amostra, ignore_index=True)
print(f"Total de registros na amostra (SIH): {len(df_sih)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.0/334.0 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.5/316.5 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.

RDSP2601.parquet: 0.00B [00:00, ?B/s]

ERSP2601.parquet: 0.00B [00:00, ?B/s]


RJSP2601.parquet: 0.00B [00:00, ?B/s]


RJSP2601.parquet:   0%|          | 0.00/676k [00:00<?, ?B/s]


RJSP2601.parquet:  10%|▉         | 65.5k/676k [00:00<00:00, 11.6MB/s]


RJSP2601.parquet:  19%|█▉        | 131k/676k [00:00<00:00, 9.13MB/s] 


RJSP2601.parquet:  29%|██▉       | 197k/676k [00:00<00:00, 7.40MB/s]


RJSP2601.parquet:  39%|███▉      | 262k/676k [00:00<00:00, 7.23MB/s]


RJSP2601.parquet:  48%|████▊     | 328k/676k [00:00<00:00, 7.06MB/s]

ERSP2601.parquet:   0%|          | 0.00/296k [00:00<?, ?B/s]

ERSP2601.parquet:  22%|██▏       | 65.5k/296k [00:00<00:00, 10.1MB/s]

ERSP2601.parquet:  44%|████▍     | 131k/296k [00:00<00:00, 6.45MB/s] 


RJSP2601.parquet:  58%|█████▊    | 393k/676k [00:00<00:00, 5.15MB/s]


RJSP2601.parquet:  68%|██████▊   | 459k/676k [00:00<00:00, 5.08MB/s]

ERSP2601.parquet:  66%|██████▋   | 197k/296k [00:00<00:00, 4.37MB/s]


RJSP2601.parquet:  78%|███████▊  | 524k/676k 

Total de registros na amostra (SIH): 1481162


In [ ]:
hospitais_fato = set(df_sih['CNES'].astype(str).str.strip())
hospitais_cnes = set(df_cnes_sp['CO_CNES'].astype(str))

sem_cnes = hospitais_fato - hospitais_cnes
print(f"Hospitais únicos no SIH: {len(hospitais_fato)}")
print(f"Com correspondência no cadastro CNES: {len(hospitais_fato & hospitais_cnes)}")
print(f"Sem correspondência: {len(sem_cnes)} ({len(sem_cnes)/len(hospitais_fato)*100:.2f}%)")

Hospitais únicos no SIH: 626
Com correspondência no cadastro CNES: 606
Sem correspondência: 20 (3.19%)


**Conclusão do cruzamento:** cobertura de 96,5% — taxa saudável, com o residual provavelmente atribuível a fechamentos ou mudanças de código CNES entre a extração do SIH e a extração do cadastro de estabelecimentos (fontes atualizadas em momentos diferentes).

## Dicionário de dados — variáveis selecionadas (Cadastro CNES)

| Coluna | Descrição |
|---|---|
| `CO_CNES` | Código do estabelecimento (chave) |
| `CO_IBGE` | Município |
| `NO_FANTASIA` / `NO_RAZAO_SOCIAL` | Nome do estabelecimento |
| `DS_ESFERA_ADMINISTRATIVA` | Municipal, Estadual, Dupla ou Sem (desabilitado) |
| `TP_GESTAO` | M/E/D/S — ver esclarecimento na Seção 4 |
| `TP_UNIDADE` | Tipo de unidade (código numérico, 39 valores distintos) |
| `DS_TURNO_ATENDIMENTO` | Turno de funcionamento |
| `ST_ATEND_HOSPITALAR` | Indica atendimento hospitalar (0/1/vazio — ver esclarecimento) |
| `CO_AMBULATORIAL_SUS` | Se atende ambulatorialmente pelo SUS |
| `NU_LATITUDE` / `NU_LONGITUDE` | Geolocalização (12% sem preenchimento) |

## Mini KPIs — Estabelecimentos

In [ ]:
print(f"Total de estabelecimentos em SP: {len(df_cnes_sp)}")
print(f"% que atendem SUS (ambulatorial): {(df_cnes_sp['CO_AMBULATORIAL_SUS'] == 'SIM').mean()*100:.1f}%")
print(f"% com atendimento hospitalar: {(df_cnes_sp['ST_ATEND_HOSPITALAR'].astype(str).isin(['1', '1.0'])).mean()*100:.1f}%")
print(f"\nDistribuição por esfera administrativa:")
print(df_cnes_sp['DS_ESFERA_ADMINISTRATIVA'].value_counts(normalize=True).mul(100).round(1))

Total de estabelecimentos em SP: 153366
% que atendem SUS (ambulatorial): 8.8%
% com atendimento hospitalar: 0.9%

Distribuição por esfera administrativa:
DS_ESFERA_ADMINISTRATIVA
MUNICIPAL    99.4
ESTADUAL      0.5
DUPLA         0.1
SEM           0.0
Name: proportion, dtype: float64


## Conclusão da exploração — NB3

A exploração do cadastro CNES em JSON resultou em: (1) confirmação de que a estrutura semiestruturada é adequada para essa fonte, já que os atributos variam de relevância entre tipos de estabelecimento; (2) investigação e explicação de dois achados aparentemente inconsistentes (`TP_GESTAO='S'` e `ST_ATEND_HOSPITALAR` vazio), ambos ligados à natureza do estabelecimento (desabilitado ou não-hospitalar), não a erro de dado; (3) validação de cobertura de 96,5% contra o SIH. Essa fonte é a base planejada para o componente de documento (JSON) da arquitetura final do projeto.